# LeetCode 668 — Kth Smallest Number in Multiplication Table bằng Python

- Link: https://leetcode.com/problems/kth-smallest-number-in-multiplication-table/
- Độ khó: **Hard**
- Chủ đề: Tìm kiếm nhị phân trên đáp án (binary search on answer), Đếm, Heap, Ma trận đã sắp theo hàng/cột

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findKthNumber(self, m: int, n: int, k: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Đây là bài mẫu mực cho kỹ thuật **tìm kiếm nhị phân trên đáp án**: thay vì tìm phần tử thứ `k` trực tiếp, ta hỏi câu dễ hơn — "**có bao nhiêu số `<= x`**?" — rồi tìm `x` nhỏ nhất mà câu trả lời `>= k`. Kỹ thuật này dùng lại ở rất nhiều bài: Kth Smallest Element in a Sorted Matrix (378), Find K-th Smallest Pair Distance (719), Koko Eating Bananas (875)...

Sau notebook này bạn sẽ:

1. Hiểu cấu trúc bảng cửu chương `m × n` và vì sao không thể sinh hết bảng khi `m, n = 3·10^4`.
2. Viết lời giải vét cạn (sinh + sắp xếp) và lời giải heap (trộn `m` hàng đã sắp).
3. Biết đếm số phần tử `<= x` trong bảng bằng công thức `Σ min(x // i, n)`.
4. Viết binary search trên đáp án đúng chuẩn (không lặp vô hạn, không lệch 1) và hiểu vì sao kết quả **luôn nằm trong bảng**.
5. Biết thêm cách đếm "bậc thang" `O(m + n)` dùng cho mọi ma trận sắp theo hàng và cột.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Bảng cửu chương kích thước `m × n` có ô hàng `i`, cột `j` (đánh số từ 1) chứa giá trị `i · j`. Cho `m`, `n`, `k`, trả về **số nhỏ thứ `k`** trong bảng (đếm cả các giá trị trùng nhau).

**Ví dụ 1**

```text
Input:  m = 3, n = 3, k = 5
Bảng:   1 2 3
        2 4 6
        3 6 9
Dãy đã sắp: 1, 2, 2, 3, 3, 4, 6, 6, 9  -> phần tử thứ 5 là 3
Output: 3
```

**Ví dụ 2**

```text
Input:  m = 2, n = 3, k = 6
Bảng:   1 2 3
        2 4 6
Dãy đã sắp: 1, 2, 2, 3, 4, 6  -> phần tử thứ 6 là 6
Output: 6
```

**Ràng buộc**

- `1 <= m, n <= 3 · 10^4`
- `1 <= k <= m · n`

Bảng có tới `9 · 10^8` ô — **không thể** tạo hết.

In [1]:
def print_table(m, n):
    for i in range(1, m + 1):
        print(' '.join(f"{i * j:>3}" for j in range(1, n + 1)))

print_table(3, 3)
vals = sorted(i * j for i in range(1, 4) for j in range(1, 4))
print("đã sắp:", vals, "| thứ 5:", vals[5 - 1])
print_table(2, 3)
vals = sorted(i * j for i in range(1, 3) for j in range(1, 4))
print("đã sắp:", vals, "| thứ 6:", vals[6 - 1])

  1   2   3
  2   4   6
  3   6   9
đã sắp: [1, 2, 2, 3, 3, 4, 6, 6, 9] | thứ 5: 3
  1   2   3
  2   4   6
đã sắp: [1, 2, 2, 3, 4, 6] | thứ 6: 6


Chú ý: "thứ `k`" đếm từ 1, nên trong list Python là `vals[k - 1]`.

## 3. Kiến thức nền

### 3.1. Phép chia nguyên `//` và câu hỏi "hàng `i` có bao nhiêu số `<= x`?"

Hàng `i` là `i, 2i, 3i, ..., n·i`. Số phần tử `<= x` là số `j` với `i·j <= x`, tức `j <= x // i`, nhưng không quá `n` → **`min(x // i, n)`**.

In [2]:
x, n = 7, 5
for i in range(1, 5):
    row = [i * j for j in range(1, n + 1)]
    print(f"hàng {i}: {row} -> số phần tử <= {x}: {sum(v <= x for v in row)} = min({x}//{i}, {n}) = {min(x // i, n)}")

hàng 1: [1, 2, 3, 4, 5] -> số phần tử <= 7: 5 = min(7//1, 5) = 5
hàng 2: [2, 4, 6, 8, 10] -> số phần tử <= 7: 3 = min(7//2, 5) = 3
hàng 3: [3, 6, 9, 12, 15] -> số phần tử <= 7: 2 = min(7//3, 5) = 2
hàng 4: [4, 8, 12, 16, 20] -> số phần tử <= 7: 1 = min(7//4, 5) = 1


### 3.2. Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Nếu điều kiện `P(x)` **đơn điệu** (sai, sai, ..., sai, đúng, đúng, ...), ta tìm `x` nhỏ nhất mà `P(x)` đúng:

```python
lo, hi = thấp, cao        # đảm bảo P(hi) đúng
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid   # mid có thể là đáp án, giữ lại
    else:      lo = mid + 1
return lo
```

In [3]:
def first_true(lo, hi, P):
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(1, 100, lambda x: x * x >= 50))     # 8 (vì 7*7=49 < 50)
print(first_true(0, 10, lambda x: x >= 0))           # 0

8
0


### 3.3. `bisect` trên list đã sắp

`bisect_right(a, x)` = số phần tử `<= x` trong list đã sắp `a`. Ta dùng nó để kiểm tra hàm đếm.

In [4]:
from bisect import bisect_right
a = [1, 2, 2, 3, 3, 4, 6, 6, 9]
print([bisect_right(a, x) for x in range(0, 10)])

[0, 1, 3, 5, 6, 6, 8, 8, 8, 9]


### 3.4. `heapq` — trộn nhiều dãy đã sắp

Mỗi hàng của bảng là một dãy tăng. Đẩy phần tử đầu của mỗi hàng vào min-heap; mỗi lần lấy ra phần tử nhỏ nhất thì đẩy phần tử kế tiếp của cùng hàng.

In [5]:
import heapq
h = [(i * 1, i, 1) for i in range(1, 4)]       # (giá trị, hàng i, cột j)
heapq.heapify(h)
out = []
for _ in range(5):
    v, i, j = heapq.heappop(h)
    out.append(v)
    if j + 1 <= 3:
        heapq.heappush(h, (i * (j + 1), i, j + 1))
print(out)

[1, 2, 2, 3, 3]


### 3.5. Đếm "bậc thang" trên ma trận sắp theo hàng và cột

Trong ma trận mà mỗi hàng tăng sang phải và mỗi cột tăng xuống dưới, đếm số phần tử `<= x` có thể đi từ **góc dưới-trái**: nếu ô hiện tại `<= x` thì cả cột phía trên đều `<= x` → cộng chiều cao, sang phải; ngược lại đi lên. Tổng số bước `<= m + n`.

In [6]:
def staircase_count(mat, x):
    rows, cols = len(mat), len(mat[0])
    r, c, cnt = rows - 1, 0, 0
    while r >= 0 and c < cols:
        if mat[r][c] <= x:
            cnt += r + 1
            c += 1
        else:
            r -= 1
    return cnt

mat = [[i * j for j in range(1, 4)] for i in range(1, 4)]
print([staircase_count(mat, x) for x in range(0, 10)])

[0, 1, 3, 5, 6, 6, 8, 8, 8, 9]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đổi câu hỏi

"Số nhỏ thứ `k` là bao nhiêu?" khó. "Có bao nhiêu số `<= x`?" dễ: `count(x) = Σ_{i=1..m} min(x // i, n)` — chỉ `O(m)` phép.

Hàm `count(x)` **không giảm** theo `x`. Số nhỏ thứ `k` chính là **`x` nhỏ nhất mà `count(x) >= k`**.

### 4.2. Tính tay ví dụ 1 (`m = n = 3, k = 5`)

| x | min(x//1,3) | min(x//2,3) | min(x//3,3) | count(x) | `>= 5`? |
|---|---|---|---|---|---|
| 1 | 1 | 0 | 0 | 1 | không |
| 2 | 2 | 1 | 0 | 3 | không |
| 3 | 3 | 1 | 1 | 5 | **có** |
| 4 | 3 | 2 | 1 | 6 | có |
| 5 | 3 | 2 | 1 | 6 | có |

`x` nhỏ nhất có `count >= 5` là `3` ✓.

### 4.3. Đáp án có luôn nằm trong bảng không?

Có. Gọi `x*` là `x` nhỏ nhất với `count(x*) >= k`. Vì `count(x* - 1) < k <= count(x*)`, nên `count` **tăng** khi đi từ `x* - 1` lên `x*` → có ít nhất một ô bằng đúng `x*`. Ta không cần kiểm tra thêm.

In [7]:
def count_le(x, m, n):
    return sum(min(x // i, n) for i in range(1, m + 1))

m, n, k = 3, 3, 5
for x in range(1, 10):
    c = count_le(x, m, n)
    print(f"x={x}: count={c}{'  <- x nhỏ nhất có count >= k' if c >= k and count_le(x - 1, m, n) < k else ''}")

x=1: count=1
x=2: count=3
x=3: count=5  <- x nhỏ nhất có count >= k
x=4: count=6
x=5: count=6
x=6: count=8
x=7: count=8
x=8: count=8
x=9: count=9


### 4.4. Nhìn `count(x)` như một hàm bậc thang

Vẽ `count(x)` theo `x`: hàm đứng yên ở các `x` không có trong bảng, và **nhảy lên** đúng bằng số lần xuất hiện của `x` khi `x` có trong bảng. Phần tử thứ `k` là chỗ đường bậc thang lần đầu chạm hoặc vượt mức `k`.

Cell dưới in "biểu đồ chữ" của `count(x)` cho bảng `3 × 4`, đánh dấu các bậc nhảy.

In [8]:
m, n = 3, 4
table_vals = sorted(i * j for i in range(1, m + 1) for j in range(1, n + 1))
print("dãy đã sắp:", table_vals)
for x in range(1, m * n + 1):
    c = count_le(x, m, n)
    jump = c - count_le(x - 1, m, n)
    bar = '#' * c
    note = f"  (+{jump}: {x} xuất hiện {jump} lần)" if jump else "  (không có trong bảng)"
    print(f"x={x:>2} count={c:>2} {bar:<12}{note}")

dãy đã sắp: [1, 2, 2, 3, 3, 4, 4, 6, 6, 8, 9, 12]
x= 1 count= 1 #             (+1: 1 xuất hiện 1 lần)
x= 2 count= 3 ###           (+2: 2 xuất hiện 2 lần)
x= 3 count= 5 #####         (+2: 3 xuất hiện 2 lần)
x= 4 count= 7 #######       (+2: 4 xuất hiện 2 lần)
x= 5 count= 7 #######       (không có trong bảng)
x= 6 count= 9 #########     (+2: 6 xuất hiện 2 lần)
x= 7 count= 9 #########     (không có trong bảng)
x= 8 count=10 ##########    (+1: 8 xuất hiện 1 lần)
x= 9 count=11 ###########   (+1: 9 xuất hiện 1 lần)
x=10 count=11 ###########   (không có trong bảng)
x=11 count=11 ###########   (không có trong bảng)
x=12 count=12 ############  (+1: 12 xuất hiện 1 lần)


Quan sát: với `k = 7`, đường bậc thang lần đầu đạt `>= 7` tại `x = 4` (count nhảy từ 5 lên 7). Thật vậy `table_vals[6] = 4`. Các `x = 5, 7, 10, 11` không có trong bảng — count đứng yên — nên binary search **không bao giờ dừng** ở chúng.

### 4.5. Vì sao không cần lo tràn số?

Trong C++/Java, `m · n` tới `9·10^8` vẫn vừa `int` 32-bit, nhưng `lo + hi` có thể tới `1,8·10^9` — sát giới hạn `2^31 - 1 ≈ 2,1·10^9`, nên người ta hay viết `lo + (hi - lo) / 2`. Python có số nguyên vô hạn nên `(lo + hi) // 2` luôn an toàn; tuy vậy biết thói quen này giúp bạn chuyển code sang ngôn ngữ khác.

In [9]:
lo, hi = 1, 30000 * 30000
print("lo + hi =", lo + hi, "| 2^31 - 1 =", 2**31 - 1)
print("mid hai cách:", (lo + hi) // 2, lo + (hi - lo) // 2)

lo + hi = 900000001 | 2^31 - 1 = 2147483647
mid hai cách: 450000000 450000000


## 5. Cách 1 — Brute Force: sinh cả bảng rồi sắp xếp

**Ý tưởng.** Tạo list mọi `i · j`, sắp xếp, lấy phần tử `k - 1`.

In [10]:
def kth_brute(m, n, k):
    vals = sorted(i * j for i in range(1, m + 1) for j in range(1, n + 1))
    return vals[k - 1]

print(kth_brute(3, 3, 5), kth_brute(2, 3, 6))
assert kth_brute(3, 3, 5) == 3
assert kth_brute(2, 3, 6) == 6

3 6


**Độ phức tạp.** Time `O(mn log(mn))`, Space `O(mn)`. Với `m = n = 3·10^4` là `9·10^8` phần tử (~7 GB bộ nhớ chỉ cho list số nguyên) — **không thể**.

**Nút thắt.** Ta tạo ra và sắp xếp **mọi** phần tử, trong khi chỉ cần biết **vị trí** của một giá trị.

In [11]:
import time
for size in (300, 600, 1000):
    t = time.perf_counter()
    kth_brute(size, size, size * size // 2)
    print(f"m=n={size}: {time.perf_counter() - t:.3f}s")

m=n=300: 0.010s
m=n=600: 0.044s


m=n=1000: 0.153s


## 6. Bước đột phá

**Insight 1 (hàng đã sắp).** Mỗi hàng tăng dần → có thể **trộn** `m` hàng như Merge k Sorted Lists, lấy ra `k` phần tử đầu (Cách 2). Tốt hơn nhưng `k` có thể tới `9·10^8`.

**Insight 2 (đếm thay vì liệt kê).** `count(x)` tính được trong `O(m)` mà không cần liệt kê. `count` đơn điệu → **binary search trên giá trị** `x ∈ [1, m·n]` (Cách 3). Chỉ cần `log2(9·10^8) ≈ 30` lần đếm.

**Invariant của binary search:** luôn có `count(lo - 1) < k` và `count(hi) >= k` (đáp án nằm trong `[lo, hi]`).

**Quy tắc nếu…thì…:**
- Nếu `count(mid) >= k` thì đáp án `<= mid` → `hi = mid`.
- Nếu `count(mid) < k` thì đáp án `> mid` → `lo = mid + 1`.
- Khi `lo == hi` thì đó là đáp án (và chắc chắn có trong bảng — mục 4.3).

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Sinh + sắp xếp (mục 5) | `O(mn log mn)` | `O(mn)` |
| 2 | Heap trộn `m` hàng | `O(m + k log m)` | `O(m)` |
| 3 | Binary search + đếm theo hàng | `O(m log(mn))` | `O(1)` |
| 4 | Binary search + đếm bậc thang | `O((m + n) log(mn))` | `O(1)` |

### 7.1. Cách 2 — Heap trộn các hàng

**Giải thích.** Heap chứa `(giá trị, hàng i, cột j)`; ban đầu là cột 1 của mọi hàng. Pop `k` lần; mỗi lần pop `(v, i, j)` thì push `(i·(j+1), i, j+1)` nếu `j < n`. Lần pop thứ `k` là đáp án.

Mẹo: hoán đổi để `m <= n` (bảng đối xứng) → heap nhỏ hơn.

In [12]:
import heapq

def kth_heap(m, n, k):
    if m > n:
        m, n = n, m
    heap = [(i, i, 1) for i in range(1, m + 1)]     # cột 1: giá trị = i
    heapq.heapify(heap)
    for _ in range(k - 1):
        v, i, j = heapq.heappop(heap)
        if j < n:
            heapq.heappush(heap, (i * (j + 1), i, j + 1))
    return heap[0][0]

print(kth_heap(3, 3, 5), kth_heap(2, 3, 6), kth_heap(3, 2, 6))
assert kth_heap(3, 3, 5) == 3

3 6 6


**Đọc code:**

- Pop `k - 1` lần, rồi phần tử nhỏ nhất còn lại (`heap[0]`) là thứ `k`.
- `if j < n`: hàng `i` còn phần tử ở cột `j + 1`.
- Heap không bao giờ rỗng khi `k <= m·n`.

**Code nộp LeetCode (Cách 2):** `k` tới `9·10^8` → **TLE**. Chỉ dùng khi `k` nhỏ.

In [13]:
from typing import List

class SolutionHeap:
    def findKthNumber(self, m: int, n: int, k: int) -> int:
        if m > n:
            m, n = n, m
        heap = [(i, i, 1) for i in range(1, m + 1)]
        heapq.heapify(heap)
        for _ in range(k - 1):
            v, i, j = heap[0]
            if j < n:
                heapq.heapreplace(heap, (v + i, i, j + 1))   # i*(j+1) = v + i
            else:
                heapq.heappop(heap)
        return heap[0][0]

assert SolutionHeap().findKthNumber(3, 3, 5) == 3
assert SolutionHeap().findKthNumber(2, 3, 6) == 6
print("SolutionHeap OK")

SolutionHeap OK


Để ý mẹo `i·(j+1) = v + i` — tránh phép nhân, và `heapreplace` thay `pop + push`.

**Độ phức tạp Cách 2:** Time `O(m + k log m)`, Space `O(m)`.

### 7.2. Cách 3 — Binary search trên đáp án + đếm theo hàng

**Giải thích.** Như mục 6. `lo = 1`, `hi = m·n` (chắc chắn `count(m·n) = m·n >= k`).

Tối ưu đếm: với `i > x` thì `x // i = 0` → chỉ duyệt `i` tới `min(m, x)`. Và với `i <= x // n` thì `min(x // i, n) = n` → cộng gộp `n · (x // n)`.

In [14]:
def count_le_fast(x, m, n):
    full = min(m, x // n)               # các hàng i <= x//n đều có đủ n phần tử <= x
    cnt = full * n
    for i in range(full + 1, min(m, x) + 1):
        cnt += x // i
    return cnt

def kth_binary(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le_fast(mid, m, n) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(kth_binary(3, 3, 5), kth_binary(2, 3, 6), kth_binary(30000, 30000, 450000000))
assert kth_binary(3, 3, 5) == 3 and kth_binary(2, 3, 6) == 6
for x in range(0, 50):
    assert count_le_fast(x, 4, 7) == count_le(x, 4, 7)

3 6 168028638


**Đọc code:**

- `full = min(m, x // n)`: hàng `i` có đủ `n` phần tử `<= x` ⇔ `i·n <= x` ⇔ `i <= x // n`.
- Các hàng còn lại `i` từ `full + 1` tới `min(m, x)`: lúc này `x // i < n` nên `min` chính là `x // i`.
- Hàng `i > x` đóng góp 0, bỏ qua.
- Binary search dạng "tìm giá trị nhỏ nhất thỏa" — `hi = mid` (không `mid - 1`) vì `mid` có thể là đáp án.

**Code nộp LeetCode (Cách 3):**

In [15]:
class SolutionBinary:
    def findKthNumber(self, m: int, n: int, k: int) -> int:
        if m > n:
            m, n = n, m                     # duyệt theo chiều nhỏ hơn
        lo, hi = 1, m * n
        while lo < hi:
            mid = (lo + hi) // 2
            full = min(m, mid // n)
            cnt = full * n
            for i in range(full + 1, min(m, mid) + 1):
                cnt += mid // i
            if cnt >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

assert SolutionBinary().findKthNumber(3, 3, 5) == 3
assert SolutionBinary().findKthNumber(2, 3, 6) == 6
print("SolutionBinary OK")

SolutionBinary OK


**Dạng binary search thứ hai (`while lo <= hi` + biến `ans`).** Nhiều người quen dạng này hơn — kết quả giống hệt:

In [16]:
def kth_binary_v2(m, n, k):
    lo, hi, ans = 1, m * n, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if count_le_fast(mid, m, n) >= k:
            ans = mid            # mid là ứng viên, thử tìm nhỏ hơn
            hi = mid - 1
        else:
            lo = mid + 1
    return ans

assert all(kth_binary_v2(4, 6, k) == kth_brute(4, 6, k) for k in range(1, 25))
print("dạng lo <= hi cũng đúng:", [kth_binary_v2(4, 6, k) for k in range(1, 25)])

dạng lo <= hi cũng đúng: [1, 2, 2, 3, 3, 4, 4, 4, 5, 6, 6, 6, 8, 8, 9, 10, 12, 12, 12, 15, 16, 18, 20, 24]


Khác biệt: dạng `lo <= hi` cần biến `ans` và dùng `hi = mid - 1` (vì `mid` đã được lưu vào `ans`). Dạng `lo < hi` không cần biến phụ nhưng phải dùng `hi = mid`. **Đừng trộn hai dạng** — đó chính là Lỗi 2 ở mục 13.

**Độ phức tạp Cách 3:** Time `O(min(m, n) · log(mn))` ≈ `3·10^4 · 30 = 9·10^5` phép; Space `O(1)`.

### 7.3. Cách 4 — Binary search + đếm bậc thang

**Giải thích.** Thay hàm đếm bằng đi bậc thang (mục 3.5) **không cần tạo ma trận**, vì ô `(i, j)` tính được ngay là `i · j`. Bắt đầu ở hàng `m`, cột 1:

```text
i = m, j = 1, cnt = 0
while i >= 1 and j <= n:
    if i * j <= x: cnt += i; j += 1     # cả cột j từ hàng 1..i đều <= x
    else:          i -= 1
```

Cách này không dùng phép chia, tổng quát cho **mọi** ma trận sắp theo hàng và cột (ví dụ LC 378).

In [17]:
def count_staircase(x, m, n):
    i, j, cnt = m, 1, 0
    while i >= 1 and j <= n:
        if i * j <= x:
            cnt += i
            j += 1
        else:
            i -= 1
    return cnt

def kth_staircase(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count_staircase(mid, m, n) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for x in range(0, 40):
    assert count_staircase(x, 5, 6) == count_le(x, 5, 6)
print(kth_staircase(3, 3, 5), kth_staircase(2, 3, 6))
assert kth_staircase(3, 3, 5) == 3

3 6


**Đọc code:**

- Ô hiện tại `(i, j)`: nếu `i·j <= x` thì mọi ô `(1..i, j)` cũng `<= x` (cột tăng xuống dưới) → cộng `i`, sang cột tiếp.
- Ngược lại `i·j > x` thì mọi ô `(i, j..n)` cũng `> x` → hàng `i` xong, đi lên.
- Mỗi bước hoặc `j` tăng hoặc `i` giảm → tối đa `m + n` bước.

**Code nộp LeetCode (Cách 4):**

In [18]:
class SolutionStaircase:
    def findKthNumber(self, m: int, n: int, k: int) -> int:
        def count(x):
            i, j, cnt = m, 1, 0
            while i >= 1 and j <= n:
                if i * j <= x:
                    cnt += i
                    j += 1
                else:
                    i -= 1
            return cnt

        lo, hi = 1, m * n
        while lo < hi:
            mid = (lo + hi) // 2
            if count(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

assert SolutionStaircase().findKthNumber(3, 3, 5) == 3
print("SolutionStaircase OK")

SolutionStaircase OK


**Độ phức tạp Cách 4:** Time `O((m + n) log(mn))`, Space `O(1)`. Chậm hơn Cách 3 khoảng 2 lần trong Python (vòng `while` nhiều bước hơn), nhưng tổng quát hơn.

## 8. TRACE từng bước (Cách 3)

Trace `m = 3, n = 3, k = 5`:

| vòng | lo | hi | mid | count(mid) | so với k=5 | hành động |
|---|---|---|---|---|---|---|
| 1 | 1 | 9 | 5 | 6 | `>=` | hi = 5 |
| 2 | 1 | 5 | 3 | 5 | `>=` | hi = 3 |
| 3 | 1 | 3 | 2 | 3 | `<` | lo = 3 |
| — | 3 | 3 | | | | dừng, trả 3 |

In [19]:
def trace_binary(m, n, k):
    lo, hi = 1, m * n
    step = 0
    print(f"{'vòng':>4} | {'lo':>6} | {'hi':>6} | {'mid':>6} | {'count':>6} | hành động")
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        c = count_le_fast(mid, m, n)
        if c >= k:
            action = f"count >= {k} -> hi = {mid}"
        else:
            action = f"count < {k}  -> lo = {mid + 1}"
        print(f"{step:>4} | {lo:>6} | {hi:>6} | {mid:>6} | {c:>6} | {action}")
        if c >= k:
            hi = mid
        else:
            lo = mid + 1
    print(f"dừng: lo = hi = {lo}")
    return lo

assert trace_binary(3, 3, 5) == 3

vòng |     lo |     hi |    mid |  count | hành động
   1 |      1 |      9 |      5 |      6 | count >= 5 -> hi = 5
   2 |      1 |      5 |      3 |      5 | count >= 5 -> hi = 3
   3 |      1 |      3 |      2 |      3 | count < 5  -> lo = 3
dừng: lo = hi = 3


Trace thêm một ví dụ lớn hơn để thấy số vòng chỉ cỡ `log2(m·n)`:

In [20]:
r = trace_binary(45, 12, 471)
assert r == kth_brute(45, 12, 471)
print("đáp án brute:", kth_brute(45, 12, 471))

vòng |     lo |     hi |    mid |  count | hành động
   1 |      1 |    540 |    270 |    444 | count < 471  -> lo = 271
   2 |    271 |    540 |    405 |    514 | count >= 471 -> hi = 405
   3 |    271 |    405 |    338 |    485 | count >= 471 -> hi = 338
   4 |    271 |    338 |    304 |    466 | count < 471  -> lo = 305
   5 |    305 |    338 |    321 |    477 | count >= 471 -> hi = 321
   6 |    305 |    321 |    313 |    472 | count >= 471 -> hi = 313
   7 |    305 |    313 |    309 |    469 | count < 471  -> lo = 310
   8 |    310 |    313 |    311 |    470 | count < 471  -> lo = 312
   9 |    312 |    313 |    312 |    472 | count >= 471 -> hi = 312
dừng: lo = hi = 312
đáp án brute: 312


Và trace hàm đếm bậc thang cho `x = 4` trên bảng `3 × 3` để thấy đường đi:

In [21]:
def trace_staircase(x, m, n):
    i, j, cnt = m, 1, 0
    while i >= 1 and j <= n:
        if i * j <= x:
            cnt += i
            print(f"ô ({i},{j}) = {i * j} <= {x}: cộng cả cột ({i} ô) -> cnt = {cnt}, sang phải")
            j += 1
        else:
            print(f"ô ({i},{j}) = {i * j} >  {x}: đi lên")
            i -= 1
    return cnt

assert trace_staircase(4, 3, 3) == 6

ô (3,1) = 3 <= 4: cộng cả cột (3 ô) -> cnt = 3, sang phải
ô (3,2) = 6 >  4: đi lên
ô (2,2) = 4 <= 4: cộng cả cột (2 ô) -> cnt = 5, sang phải
ô (2,3) = 6 >  4: đi lên
ô (1,3) = 3 <= 4: cộng cả cột (1 ô) -> cnt = 6, sang phải


## 9. Code nộp LeetCode

Lời giải chính: **Binary search trên đáp án + đếm theo hàng** (Cách 3).

In [22]:
class Solution:
    def findKthNumber(self, m: int, n: int, k: int) -> int:
        if m > n:                       # bảng đối xứng: duyệt chiều nhỏ hơn
            m, n = n, m

        def count(x: int) -> int:
            """Số ô có giá trị <= x."""
            full = min(m, x // n)       # các hàng có đủ n ô <= x
            cnt = full * n
            for i in range(full + 1, min(m, x) + 1):
                cnt += x // i
            return cnt

        lo, hi = 1, m * n               # count(hi) = m*n >= k
        while lo < hi:
            mid = (lo + hi) // 2
            if count(mid) >= k:
                hi = mid                # mid có thể là đáp án
            else:
                lo = mid + 1
        return lo

sol = Solution()
assert sol.findKthNumber(3, 3, 5) == 3
assert sol.findKthNumber(2, 3, 6) == 6
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (hàm đếm đúng).** Hàng `i` gồm `i·1 < i·2 < ... < i·n`. Số phần tử `<= x` là số `j ∈ [1, n]` với `j <= x / i`, tức `min(⌊x/i⌋, n)`. Cộng qua các hàng được `count(x)`.

**Bổ đề 2 (đơn điệu).** `x <= y` ⇒ `count(x) <= count(y)` (mỗi hạng `min(⌊x/i⌋, n)` không giảm).

**Bổ đề 3 (định nghĩa phần tử thứ `k`).** Gọi `a_1 <= a_2 <= ... <= a_{mn}` là dãy đã sắp. Khi đó `a_k` là `x` nhỏ nhất với `count(x) >= k`:
- `count(a_k) >= k` vì `a_1..a_k` đều `<= a_k`.
- Với `x < a_k`: mọi phần tử `<= x` đều `< a_k`, nên nằm trong `a_1..a_{k-1}` → `count(x) <= k - 1`.

**Binary search đúng.** Bất biến: đáp án `x*` thuộc `[lo, hi]`. Ban đầu `[1, mn]` chứa `x*`. Nếu `count(mid) >= k` thì `x* <= mid` → `hi = mid` giữ bất biến; ngược lại `x* > mid` → `lo = mid + 1`. Khoảng thu hẹp mỗi vòng (vì `mid < hi`), dừng khi `lo == hi == x*`.

**Đáp án nằm trong bảng.** Theo bổ đề 3, `x* = a_k` là một giá trị của bảng — không cần kiểm tra thêm.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Sinh + sắp | `O(mn log mn)` | `O(mn)` |
| 2. Heap | `O(m + k log m)` | `O(m)` |
| 3. Binary + hàng | `O(min(m,n) · log(mn))` | `O(1)` |
| 4. Binary + bậc thang | `O((m+n) · log(mn))` | `O(1)` |

Với `m = n = 3·10^4`: Cách 3 cỡ `30 · 3·10^4 = 9·10^5` phép chia — dưới 0,5 giây trong Python.

In [23]:
for name, f in [("binary", kth_binary), ("staircase", kth_staircase), ("Solution", Solution().findKthNumber)]:
    t = time.perf_counter()
    r = f(30000, 30000, 450000000)
    print(f"{name:>10}: {r}  {time.perf_counter() - t:.3f}s")
t = time.perf_counter()
print("heap k nhỏ:", kth_heap(30000, 30000, 1000), f"{time.perf_counter() - t:.3f}s")

    binary: 168028638  0.023s


 staircase: 168028638  0.100s
  Solution: 168028638  0.023s
heap k nhỏ: 186 0.005s


## 12. Bộ test

Gồm: ví dụ đề, edge case (`k = 1`, `k = m·n`, `m = 1`, `n = 1`, `m > n`) và đối chiếu ngẫu nhiên có seed giữa các cách với brute force.

In [24]:
def all_methods(m, n, k):
    return {
        "heap": kth_heap(m, n, k),
        "binary": kth_binary(m, n, k),
        "staircase": kth_staircase(m, n, k),
        "SolutionHeap": SolutionHeap().findKthNumber(m, n, k),
        "SolutionBinary": SolutionBinary().findKthNumber(m, n, k),
        "SolutionStaircase": SolutionStaircase().findKthNumber(m, n, k),
        "Solution": Solution().findKthNumber(m, n, k),
    }

cases = [
    ((3, 3, 5), 3), ((2, 3, 6), 6), ((1, 1, 1), 1),
    ((1, 10, 7), 7), ((10, 1, 7), 7),       # một hàng / một cột
    ((5, 5, 1), 1), ((5, 5, 25), 25),       # k nhỏ nhất / lớn nhất
    ((4, 9, 20), kth_brute(4, 9, 20)), ((9, 4, 20), kth_brute(9, 4, 20)),
]
for (m, n, k), want in cases:
    assert kth_brute(m, n, k) == want
    for name, got in all_methods(m, n, k).items():
        assert got == want, ((m, n, k), name, got, want)
print("Tất cả", len(cases), "test cố định OK")

Tất cả 9 test cố định OK


In [25]:
import random
random.seed(2024)
for trial in range(500):
    m, n = random.randint(1, 25), random.randint(1, 25)
    k = random.randint(1, m * n)
    want = kth_brute(m, n, k)
    for name, got in all_methods(m, n, k).items():
        assert got == want, ((m, n, k), name, got, want)
print("500 test ngẫu nhiên: mọi cách khớp brute force")

500 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên giới hạn `n` trong hàm đếm

**Code sai:** `count = sum(x // i for i in range(1, m + 1))`.

**Vì sao sai:** hàng `i` chỉ có `n` phần tử; `x // i` có thể vượt `n` (ví dụ `i = 1`, `x` lớn) → đếm thừa.

In [26]:
def kth_wrong_count(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // i for i in range(1, m + 1)) >= k:      # SAI: thiếu min(..., n)
            hi = mid
        else:
            lo = mid + 1
    return lo

print("sai :", kth_wrong_count(2, 3, 6), "| đúng:", Solution().findKthNumber(2, 3, 6))

sai : 4 | đúng: 6


**Code đúng:** `min(x // i, n)`.

### Lỗi 2: Dùng `hi = mid - 1` khi `count(mid) >= k`

**Code sai:** coi `mid` "đã dùng rồi" nên loại luôn.

**Vì sao sai:** `mid` có thể chính là đáp án (count có thể nhảy qua `k` ngay tại `mid`).

In [27]:
def kth_wrong_hi(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid, m, n) >= k:
            hi = mid - 1                 # SAI
        else:
            lo = mid + 1
    return lo

for kk in (2, 3, 5):
    print(f"k={kk}: sai :", kth_wrong_hi(3, 3, kk), "| đúng:", Solution().findKthNumber(3, 3, kk))

k=2: sai : 1 | đúng: 2
k=3: sai : 1 | đúng: 2
k=5: sai : 3 | đúng: 3


Với `k = 5` code sai "tình cờ" trả đúng, nhưng `k = 2, 3` thì sai — đây là lý do phải test nhiều giá trị `k`, không chỉ ví dụ đề.

**Code đúng:** `hi = mid` với vòng `while lo < hi`. (Hoặc dạng `while lo <= hi` có biến `ans` lưu `mid` mỗi khi `count(mid) >= k`.)

### Lỗi 3: Dừng sớm khi `count(mid) == k`

**Code sai:** `if count(mid) == k: return mid`.

**Vì sao sai:** `count(mid) == k` không có nghĩa `mid` nằm trong bảng. Ví dụ `m = n = 3, k = 6`: `count(4) = 6` và `count(5) = 6`; nếu `mid = 5` thì trả 5 — **không có trong bảng**. Ngoài ra có thể không có `x` nào với `count(x) == k` (khi có giá trị trùng), gây vòng lặp sai.

In [28]:
def kth_wrong_equal(m, n, k):
    lo, hi = 1, m * n
    while lo <= hi:
        mid = (lo + hi) // 2
        c = count_le(mid, m, n)
        if c == k:
            return mid                   # SAI: mid có thể không có trong bảng
        if c < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return lo

print("sai :", kth_wrong_equal(3, 3, 6), "| đúng:", Solution().findKthNumber(3, 3, 6))
print("count(4) =", count_le(4, 3, 3), "count(5) =", count_le(5, 3, 3))

sai : 5 | đúng: 4
count(4) = 6 count(5) = 6


**Code đúng:** luôn tìm `x` **nhỏ nhất** có `count(x) >= k`.

### Lỗi 4: Đặt `hi = max(m, n)` hoặc `hi = k`

`hi` phải đảm bảo `count(hi) >= k`. `hi = m·n` luôn đúng. (`hi = k` thực ra cũng đúng vì `count(k) >= k` — hàng 1 có `min(k, n)` phần tử, cột 1 có `min(k, m)`... nhưng chứng minh rắc rối hơn; `m·n` an toàn và chỉ thêm vài vòng.) Còn `hi = max(m, n)` thì sai khi `k` lớn.

## 14. Mẹo & nhận dạng pattern

- **"Phần tử nhỏ thứ k" trong một tập ngầm định quá lớn để liệt kê** + đếm được "số phần tử `<= x`" nhanh → **binary search trên đáp án**.
- Ma trận **sắp theo hàng và cột** → đếm bậc thang `O(m + n)`.
- Nhiều dãy đã sắp, `k` nhỏ → heap k-way merge.
- Mẫu "tìm `x` nhỏ nhất thỏa": `while lo < hi: mid = (lo+hi)//2; if P(mid): hi = mid else: lo = mid + 1`.
- Bài cùng họ: 378 (Kth Smallest in Sorted Matrix), 719 (K-th Smallest Pair Distance), 786 (K-th Smallest Prime Fraction), 1201 (Ugly Number III).

## 15. Biến thể

1. **Kth lớn nhất** — bằng phần tử nhỏ thứ `m·n - k + 1`.
2. **Kth Smallest Element in a Sorted Matrix (LC 378)** — ma trận bất kỳ sắp theo hàng/cột: binary search + bậc thang.
3. **Đếm số giá trị khác nhau `<= x`** — khó hơn nhiều (bài toán bảng cửu chương của Erdős!), không dùng được công thức đơn giản.
4. **Bảng `i + j` thay vì `i · j`** — hàm đếm `Σ clamp(x - i, 0, n)`.

In [29]:
def kth_largest(m, n, k):
    return Solution().findKthNumber(m, n, m * n - k + 1)

def kth_sorted_matrix(mat, k):
    """Biến thể 2 (LC 378)."""
    rows, cols = len(mat), len(mat[0])
    lo, hi = mat[0][0], mat[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if staircase_count(mat, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def kth_sum_table(m, n, k):
    """Biến thể 4: bảng i + j."""
    def count(x):
        return sum(max(0, min(x - i, n)) for i in range(1, m + 1))
    lo, hi = 2, m + n
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert kth_largest(3, 3, 1) == 9 and kth_largest(3, 3, 2) == 6
mat = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
assert kth_sorted_matrix(mat, 8) == 13
for m in range(1, 6):
    for n in range(1, 6):
        vals = sorted(i + j for i in range(1, m + 1) for j in range(1, n + 1))
        for k in range(1, m * n + 1):
            assert kth_sum_table(m, n, k) == vals[k - 1]
print("Biến thể OK")

Biến thể OK


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh + sắp | liệt kê hết | `O(mn log mn)` | `O(mn)` | kiểm thử với bảng nhỏ |
| 2. Heap | trộn `m` hàng, pop `k` lần | `O(m + k log m)` | `O(m)` | `k` nhỏ |
| 3. Binary + đếm hàng | `x` nhỏ nhất có `Σ min(x//i, n) >= k` | `O(min(m,n) log mn)` | `O(1)` | bản nộp chuẩn |
| 4. Binary + bậc thang | đếm bằng đi góc dưới-trái | `O((m+n) log mn)` | `O(1)` | ma trận sắp hàng/cột tổng quát |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `findKthNumber(4, 4, 7)`, `findKthNumber(1, 5, 3)`, `findKthNumber(3, 5, 15)`?

**Bài 2 (tính tay).** Tính `count(x)` với `m = 4, n = 5` cho `x = 6` bằng công thức và bằng đếm bậc thang.

**Bài 3 (điền TODO).** Hoàn thiện `count_rows(x, m, n)` (không nhìn code).

**Bài 4 (tự viết lại).** Viết lại Cách 3 `my_kth(m, n, k)`.

**Bài 5 (mini challenge).** Viết `kth_pair_distance(nums, k)` (LC 719): số nhỏ thứ `k` trong mọi `|nums[i] - nums[j]|` với `i < j`. Gợi ý: sắp `nums`, binary search trên khoảng cách, đếm bằng hai con trỏ.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_rows(x, m, n):
    # TODO: tổng min(x // i, n) với i = 1..m
    pass

# Bài 4 — TODO
def my_kth(m, n, k):
    # TODO: lo = 1, hi = m*n; tìm x nhỏ nhất có count(x) >= k
    pass

# Bài 5 — TODO
def kth_pair_distance(nums, k):
    # TODO: sắp; binary search d trong [0, max-min]; đếm số cặp có hiệu <= d bằng hai con trỏ
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: viết bảng ra và sắp.
- Mức 2: bảng 4×4: `1,2,2,3,3,4,4,4,...` → thứ 7 là 4.
- Mức 3: `(1,5,3)` → 3; `(3,5,15)` → phần tử lớn nhất = 15.

**Bài 2.**
- Mức 1: `min(6//1,5) + min(6//2,5) + min(6//3,5) + min(6//4,5)`.
- Mức 2: `5 + 3 + 2 + 1 = 11`.
- Mức 3: bậc thang bắt đầu ô `(4,1) = 4 <= 6` cộng 4, ô `(4,2) = 8 > 6` lên, ...

**Bài 3.**
- Mức 1: vòng `for i in range(1, m + 1)`.
- Mức 2: `cnt += min(x // i, n)`.
- Mức 3: có thể dừng sớm khi `i > x`.

**Bài 4.**
- Mức 1: `while lo < hi`.
- Mức 2: `mid = (lo + hi) // 2`.
- Mức 3: `if count(mid) >= k: hi = mid else: lo = mid + 1`.

**Bài 5.**
- Mức 1: `count(d)` = số cặp `i < j` với `nums[j] - nums[i] <= d`.
- Mức 2: với mỗi `j`, dời `i` sang phải tới khi `nums[j] - nums[i] <= d`, cộng `j - i`.
- Mức 3: tìm `d` nhỏ nhất có `count(d) >= k`.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert Solution().findKthNumber(4, 4, 7) == 4
assert Solution().findKthNumber(1, 5, 3) == 3
assert Solution().findKthNumber(3, 5, 15) == 15

# Bài 2
assert count_le(6, 4, 5) == 11 == count_staircase(6, 4, 5)
trace_staircase(6, 4, 5)

# Bài 3
def count_rows_ans(x, m, n):
    cnt = 0
    for i in range(1, m + 1):
        if i > x:
            break
        cnt += min(x // i, n)
    return cnt

assert all(count_rows_ans(x, 6, 7) == count_le(x, 6, 7) for x in range(0, 50))
print("Bài 1–3 OK")

ô (4,1) = 4 <= 6: cộng cả cột (4 ô) -> cnt = 4, sang phải
ô (4,2) = 8 >  6: đi lên
ô (3,2) = 6 <= 6: cộng cả cột (3 ô) -> cnt = 7, sang phải
ô (3,3) = 9 >  6: đi lên
ô (2,3) = 6 <= 6: cộng cả cột (2 ô) -> cnt = 9, sang phải
ô (2,4) = 8 >  6: đi lên
ô (1,4) = 4 <= 6: cộng cả cột (1 ô) -> cnt = 10, sang phải
ô (1,5) = 5 <= 6: cộng cả cột (1 ô) -> cnt = 11, sang phải
Bài 1–3 OK


In [32]:
# Bài 4
def my_kth_ans(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count_rows_ans(mid, m, n) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

# Bài 5
def kth_pair_distance_ans(nums, k):
    a = sorted(nums)

    def count(d):
        cnt, i = 0, 0
        for j in range(len(a)):
            while a[j] - a[i] > d:
                i += 1
            cnt += j - i
        return cnt

    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

random.seed(8)
for _ in range(300):
    m, n = random.randint(1, 15), random.randint(1, 15)
    k = random.randint(1, m * n)
    assert my_kth_ans(m, n, k) == kth_brute(m, n, k)
    nums = [random.randint(0, 30) for _ in range(random.randint(2, 9))]
    dists = sorted(abs(nums[i] - nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))
    kk = random.randint(1, len(dists))
    assert kth_pair_distance_ans(nums, kk) == dists[kk - 1]
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template binary search trên đáp án:**

```python
def count(x):            # số phần tử <= x, phải đơn điệu
    ...
lo, hi = MIN, MAX        # count(hi) >= k chắc chắn
while lo < hi:
    mid = (lo + hi) // 2
    if count(mid) >= k:
        hi = mid
    else:
        lo = mid + 1
return lo
```

**Hàm đếm bảng cửu chương:** `sum(min(x // i, n) for i in range(1, m + 1))`.

**Checklist:**

- [ ] `min(x // i, n)` — không quên giới hạn số cột.
- [ ] `hi = mid` (không `mid - 1`) khi `count(mid) >= k`.
- [ ] Không `return` khi `count(mid) == k`.
- [ ] `hi` ban đầu đủ lớn (`m · n`).
- [ ] Hoán đổi để duyệt chiều nhỏ hơn (`m <= n`).
- [ ] Thử `k = 1`, `k = m·n`, `m = 1`, `n = 1`.